### Если данные уже готовы (X готов, OHE сделан)
### 1. SGDRegressor (линейная модель)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDRegressor
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# Загрузка твоих готовых данных
df = pd.read_csv("твой_файл.csv")
y = np.log1p(df["usd_price"])          # таргет — всегда логарифм!
X = df.drop(columns=["usd_price"])
# Масштабирование обязательно для SGD
pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),  # with_mean=False если есть sparse
    ("model", SGDRegressor(
        loss="huber",
        penalty="elasticnet",
        max_iter=3000,
        random_state=0,
    ))
])
# GridSearch
grid = {
    "model__alpha":    [1e-5, 1e-4, 1e-3],
    "model__l1_ratio": [0.15, 0.5, 0.85],
    "model__epsilon":  [0.05, 0.1, 0.5],
}
cv = KFold(n_splits=5, shuffle=True, random_state=0)
gs = GridSearchCV(pipe, grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1)
gs.fit(X, y)
print("Лучшие параметры:", gs.best_params_)
print("CV RMSE (log):", -gs.best_score_)
# OOF предсказания для честной оценки
y_pred_log = cross_val_predict(gs.best_estimator_, X, y, cv=cv)
y_pred_usd = np.expm1(y_pred_log)
y_true_usd = np.expm1(y)
# Метрики
mape = np.abs((y_true_usd - y_pred_usd) / y_true_usd).mean() * 100
med_ape = np.median(np.abs((y_true_usd - y_pred_usd) / y_true_usd)) * 100
r2_log = 1 - np.sum((y - y_pred_log)**2) / np.sum((y - y.mean())**2)
print(f"R² (log): {r2_log:.4f}")
print(f"MAPE: {mape:.2f}%,  median APE: {med_ape:.2f}%")


#### 2. CatBoost
#### ⚠️ Если ты уже сделал OHE — CatBoost лучше подавать без OHE (он сам обрабатывает категории). Но если данные уже OHE — просто подавай числовую матрицу.


In [ ]:
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold, cross_val_predict
import numpy as np
df = pd.read_csv("твой_файл.csv")
y = np.log1p(df["usd_price"])
X = df.drop(columns=["usd_price"])
cb = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.03,
    l2_leaf_reg=3.0,
    loss_function="RMSE",
    random_seed=0,
    verbose=100,
)
cv = KFold(n_splits=5, shuffle=True, random_state=0)
# OOF predict
oof = np.zeros(len(df))
for train_idx, val_idx in cv.split(X, y):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr = y.iloc[train_idx]
    cb.fit(X_tr, y_tr, eval_set=(X_val, y.iloc[val_idx]), early_stopping_rounds=50)
    oof[val_idx] = cb.predict(X_val)
y_usd = np.expm1(y)
p_usd = np.expm1(oof)
med_ape = np.median(np.abs((y_usd - p_usd) / y_usd)) * 100
r2 = 1 - np.sum((y - oof)**2) / np.sum((y - y.mean())**2)
print(f"R² (log): {r2:.4f},  median APE: {med_ape:.2f}%")
# Финальная модель на всех данных
cb.fit(X, y)
cb.save_model("catboost_model.cbm")


### 3. Сравнение результатов

#### Ожидаемые ориентиры из нашего проекта:
#### SGDRegressor:  R²(log) ≈ 0.807,  median APE ≈ 13.2%
#### CatBoost:      R²(log) ≈ 0.891,  median APE ≈  9.5%
Главные правила:

y = log1p(price) — всегда, иначе выбросы сломают обучение
StandardScaler — обязателен для SGD, CatBoost не нужен
early_stopping_rounds=50 — спасает CatBoost от переобучения
Метрика для сравнения — median APE (устойчива к выбросам)
Скинь структуру своих данных если хочешь — подстрою код под твои колонки.